In [29]:
import numpy as np

* You will only be given these four data structures
* No other template code or coding-by-contract will be provided
* It may benefit you to utlize Object-Oriented Programming (OOP) as you will be developing out the complete HMM suite throughout the HMM modules
* Make no assumptions as to the number of hidden states you will be given
* Make no assumptions as to the number of distinct observations you will be given
* Make no assumptions that the data structures will be modeling CpG islands (these were just examples)

In [83]:
class HiddenMarkovModel:
    def __init__(self, initial_probs, transition_probs, emission_probs):
        self.states = []
        self.states = list(initial_probs.keys())
        self.initial_probs = initial_probs
        self.transition_probs = transition_probs
        self.emission_probs = emission_probs
    def get_states(self):
        return self.states

    def get_initial_probs(self, state):
        return self.initial_probs[state]

    def get_transition_probs(self, state):
        return self.transition_probs[state]

    def get_emission_probs(self, state):
        return self.emission_probs[state]


In [284]:
def viterbi_algorithm(observations, initial_probs, transition_probs, emission_probs):
    if type(observations) != list:
        observations = [observations]

    # Create HMM class
    hmm = HiddenMarkovModel(initial_probs, transition_probs, emission_probs)
    print(hmm.get_states())
    optimal_path = []
    for observation in observations:
        viterbi_matrix, traceback_matrix = build_viterbi_matrix(observation, hmm)
        print(f"Observation: {observation}")
        print(f"Viterbi matrix:\n{viterbi_matrix}")
        print(f"Traceback matrix:\n{traceback_matrix}")
        optimal_path_index = viterbi_traceback(viterbi_matrix, traceback_matrix)
        print(f"Optimal path index:\n{optimal_path_index}")
        path = convert_index_to_states(optimal_path_index, hmm.get_states())
        print(f"Optimal path:\n{path}\n")
        optimal_path.append(path)


    return optimal_path



In [299]:
def build_viterbi_matrix(observations, hmm):
    ####### Initialization ########

    # Get states and observations from class object
    states = hmm.get_states()

    #Initialize viterbi and traceback matrix
    prob_matrix = np.zeros((len(states), len(observations)), dtype = float)
    traceback_matrix = np.zeros((len(states), len(observations)), dtype = int)

    ####### Iteration ########
    for i, observation in enumerate(observations):
        for j, state in enumerate(states):
            state_init_probs = hmm.get_initial_probs(state)
            state_emit_probs = hmm.get_emission_probs(state)
            state_trans_probs = hmm.get_transition_probs(state)

            # If we are looking at the first observation
            if i == 0:
                # Calculate the initial probability o per state
                state_prob = state_init_probs* state_emit_probs[observation]
                # Use natural log to prevent numerical underflow
                #prob_matrix[j][i] = round(np.log(state_prob), 2)
                prob_matrix[j][i] = state_prob
            # Else if we are looking at the second observation onward
            else:
                # Calculate the possible probabilities based on transitioning for all states
                #possible_probs = [np.exp(prob_matrix[k][i-1]) * state_trans_probs[prev_state] * state_emit_probs[observation] for k, prev_state in enumerate(states)]
                possible_probs = [prob_matrix[k][i-1] * hmm.get_transition_probs(prev_state)[state] * state_emit_probs[observation] for k, prev_state in enumerate(states)]
                # Get the max probability and add the log value to the viterbi matrix
                max_prob = max(possible_probs)
                #prob_matrix[j][i] = round(np.log(max_prob), 2)
                prob_matrix[j][i] = max_prob
                # Get the index of the maximum probabilities and add that to the traceback matrix
                previous_coords = np.argmax(possible_probs)
                traceback_matrix[j][i] = previous_coords

    return prob_matrix, traceback_matrix



In [300]:
def viterbi_traceback(viterbi_matrix, traceback_matrix):

    # Identify the final state with the highest probability
    end_state = np.argmax(viterbi_matrix[:,-1])

    # Add state to list
    predictions = [int(end_state)]

    # Traceback through the matrix starting at the end of the matrix
    for i in range(len(obs)-1, 0, -1):
        # Append the state index to the predictions
        end_state = traceback_matrix[end_state][i]
        predictions.append(int(end_state))

    # Reverse the predictions so it starts at the beginning of matrix
    return predictions[::-1]

In [301]:
 def convert_index_to_states(predictions, states):
    states_final = []
    for index in predictions:
        states_final.append(states[index])
    return states_final

In [302]:

# Example observation sequence following the powerpoint
obs = "ACGCGATC"

# Example initial probabilities (probability of starting in each state)
init_probs = {
    "I": 0.1,
    "G": 0.9
}

# Example transition probabilities (probability of moving from one state to another)
trans_probs = {
    "I": {"I": 0.6, "G": 0.4},
    "G": {"I": 0.1, "G": 0.9}
}

# Example emission probabilities (probability of observing a symbol in a given state)
emit_probs = {
    "I": {"A": 0.1, "C": 0.4, "G": 0.4, "T": 0.1},
    "G": {"A": 0.4, "C": 0.1, "G": 0.1, "T": 0.4}
}
path1 = viterbi_algorithm(obs, init_probs, trans_probs, emit_probs)


['I', 'G']
Observation: ACGCGATC
Viterbi matrix:
[[1.00000000e-02 1.44000000e-02 3.45600000e-03 8.29440000e-04
  1.99065600e-04 1.19439360e-05 7.16636160e-07 4.58647142e-07]
 [3.60000000e-01 3.24000000e-02 2.91600000e-03 2.62440000e-04
  3.31776000e-05 3.18504960e-05 1.14661786e-05 1.03195607e-06]]
Traceback matrix:
[[0 1 0 0 0 0 0 1]
 [0 1 1 1 0 0 1 1]]
Optimal path index:
[1, 0, 0, 0, 0, 1, 1, 1]
Optimal path:
['G', 'I', 'I', 'I', 'I', 'G', 'G', 'G']



In [303]:
# Example observation sequence
observations = ["GGCACTGAA", "ACGCGATC"]

# Example initial probabilities (probability of starting in each state)
init_probs = {
    "CpG": 0.3,
    "Genome": 0.5,
    "Promoter": 0.2
}

# Example transition probabilities (probability of moving from one state to another)
trans_probs = {
    "CpG": {"CpG": 0.6, "Promoter": 0.2, "Genome": 0.2},
    "Genome": {"CpG": 0.2, "Promoter": 0.1, "Genome": 0.7},
    "Promoter": {"CpG": 0.1, "Promoter": 0.7, "Genome": 0.2},
}

# Example emission probabilities (probability of observing a symbol in a given state)
emit_probs = {
    "CpG": {"A": 0.1, "C": 0.4, "G": 0.4, "T": 0.1},
    "Genome": {"A": 0.3, "C": 0.2, "G": 0.2, "T": 0.3},
    "Promoter": {"A": 0.2, "C": 0.3, "G": 0.3, "T": 0.2},
}
path2 = viterbi_algorithm(observations, init_probs, trans_probs, emit_probs)

['CpG', 'Genome', 'Promoter']
Observation: GGCACTGAA
Viterbi matrix:
[[1.20000000e-01 2.88000000e-02 6.91200000e-03 4.14720000e-04
  9.95328000e-05 5.97196800e-06 1.43327232e-06 8.59963392e-08
  7.16934758e-09]
 [1.00000000e-01 1.40000000e-02 1.96000000e-03 4.14720000e-04
  5.80608000e-05 1.21927680e-05 1.70698752e-06 3.58467379e-07
  7.52781496e-08]
 [6.00000000e-02 1.26000000e-02 2.64600000e-03 3.70440000e-04
  7.77924000e-05 1.08909360e-05 2.28709656e-06 3.20193518e-07
  4.48270926e-08]]
Traceback matrix:
[[0 0 0 0 0 0 0 0 1]
 [0 1 1 0 1 1 1 1 1]
 [0 2 2 2 2 2 2 2 2]]
Optimal path index:
[0, 0, 0, 1, 1, 1, 1, 1]
Optimal path:
['CpG', 'CpG', 'CpG', 'Genome', 'Genome', 'Genome', 'Genome', 'Genome']

Observation: ACGCGATC
Viterbi matrix:
[[3.00000000e-02 1.20000000e-02 2.88000000e-03 6.91200000e-04
  1.65888000e-04 9.95328000e-06 5.97196800e-07 2.03297472e-07]
 [1.50000000e-01 2.10000000e-02 2.94000000e-03 4.11600000e-04
  5.76240000e-05 1.21010400e-05 2.54121840e-06 3.55770576e-07]
 [

In [304]:
# Example observation sequence
obs = "GGCACTGAA"

# Example initial probabilities (probability of starting in each state)
init_probs = {
    "I": 0.2,
    "G": 0.8
}

# Example transition probabilities (probability of moving from one state to another)
trans_probs = {
    "I": {"I": 0.7, "G": 0.3},
    "G": {"I": 0.1, "G": 0.9}
}

# Example emission probabilities (probability of observing a symbol in a given state)
emit_probs = {
    "I": {"A": 0.1, "C": 0.4, "G": 0.4, "T": 0.1},
    "G": {"A": 0.3, "C": 0.2, "G": 0.2, "T": 0.3}
}
path3 = viterbi_algorithm(obs, init_probs, trans_probs, emit_probs)

['I', 'G']
Observation: GGCACTGAA
Viterbi matrix:
[[8.00000000e-02 2.24000000e-02 6.27200000e-03 4.39040000e-04
  1.22931200e-04 8.60518400e-06 2.72097792e-06 1.90468454e-07
  3.30598817e-08]
 [1.60000000e-01 2.88000000e-02 5.18400000e-03 1.39968000e-03
  2.51942400e-04 6.80244480e-05 1.22444006e-05 3.30598817e-06
  8.92616807e-07]]
Traceback matrix:
[[0 0 0 0 0 0 1 0 1]
 [0 1 1 1 1 1 1 1 1]]
Optimal path index:
[1, 1, 1, 1, 1, 1, 1, 1, 1]
Optimal path:
['G', 'G', 'G', 'G', 'G', 'G', 'G', 'G', 'G']

